In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from tqdm import tqdm

############################################
# 1) Алфавит
############################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
num_classes = len(alphabet)  # 45 (0..44, где 0 = <pad>/blank)

############################################
# 2) encode_label
############################################
def encode_label(text: str, alpha: dict) -> list:
    indices = []
    for ch in text:
        if ch in alpha:
            indices.append(alpha[ch])
    return indices

############################################
# 3) Преобразование audio -> более детальный Mel
#    (без аугментаций)
############################################
def audio_to_melspectrogram(y, sr=16000, n_mels=128, n_fft=1024, hop_length=256):
    """
    n_mels=128  -> увеличиваем частотное разрешение (если позволяет память).
    hop_length=256 -> более частая дискретизация по времени (увеличивает размер time).
    """
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    log_S = librosa.power_to_db(S, ref=np.max)
    # Нормируем 0..1
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm  # [n_mels=128, time]

############################################
# 4) Датасет (без аугментаций)
############################################
class MorseAudioCTCDataset(Dataset):
    def __init__(self, df, sr=16000, transform=None):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        audio_path = "morse_dataset/" + row['id']  # или как у вас
        y, _ = librosa.load(audio_path, sr=self.sr)

        # Превращаем в Mel
        if self.transform:
            mel = self.transform(y, sr=self.sr)
        else:
            mel = y  # fallback, raw

        mel_tensor = torch.tensor(mel, dtype=torch.float)

        # Кодируем label
        text_label = row['message']
        label_indices = encode_label(text_label, alphabet)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)

        time_dim = mel_tensor.shape[1]
        return mel_tensor, label_tensor, time_dim

############################################
# 5) Collate_fn
############################################
def ctc_collate_fn(batch, pool_time_factor=8):
    """
    Здесь pool_time_factor=8, т.к. будет 3 блока Conv+Pool, что даёт уменьшение time в 2*2*2=8 раз.
    Если будет только 2 пула, то ставь 4.
    """
    mel_list = []
    label_list = []
    tgt_len_list = []
    time_list = []

    for (mel, label, tdim) in batch:
        mel_list.append(mel)
        label_list.append(label)
        tgt_len_list.append(len(label))
        time_list.append(tdim)

    max_time = max(m.shape[1] for m in mel_list)
    padded_mels = []
    for mel in mel_list:
        diff = max_time - mel.shape[1]
        if diff > 0:
            mel = F.pad(mel, (0, diff), value=0.0)
        mel = mel.unsqueeze(0)  # => [1, n_mels, max_time]
        padded_mels.append(mel)

    audio_batch = torch.stack(padded_mels, dim=0)  # [B, 1, n_mels, max_time]
    labels_concat = torch.cat(label_list, dim=0)

    input_lengths = []
    for tdim in time_list:
        input_lengths.append(tdim // pool_time_factor)
    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(tgt_len_list, dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

############################################
# 6) Сложная модель (3 Conv+Pool, BiLSTM)
############################################
class DeepCTCModel(nn.Module):
    """
    - Три Conv-блока (Conv->BN->ReLU->Pool), уменьшая time/freq в 2 раза каждый раз => /8
    - BiLSTM (2-3 слоя, hidden=256)
    - Dropout в Conv-блоках и/или LSTM
    - Выход Linear
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=128,
                 hidden_size=256, lstm_layers=2, dropout=0.2):
        super(DeepCTCModel, self).__init__()

        # Блок 1
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2))  # /2 time /2 freq
        )
        # Блок 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2))  # /2
        )
        # Блок 3
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2))  # /2
        )
        # Итого после 3 пулов: time/freq => /8, channels=128
        # => LSTM input = 128*(n_mels//8)

        self.lstm = nn.LSTM(
            input_size=128 * (n_mels // 8),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=(dropout if lstm_layers>1 else 0.0),
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size*2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels=128, time=?]
        => output: [time//8, B, num_classes]
        """
        x = self.conv1(x)  # -> [B, 32, n_mels//2, time//2]
        x = self.conv2(x)  # -> [B, 64, n_mels//4, time//4]
        x = self.conv3(x)  # -> [B, 128, n_mels//8, time//8]
        b,c,f,t = x.shape

        # Склеим (channels*f) => feature
        x = x.view(b, c*f, t)
        # [B, 128*(n_mels//8), time//8]
        x = x.permute(2,0,1)  # [time//8, B, features]

        lstm_out, _ = self.lstm(x)  # [time//8, B, hidden_size*2]
        logits = self.fc(lstm_out)  # [time//8, B, num_classes]

        return logits

############################################
# 7) Цикл обучения
############################################
def train_ctc_loop(model, train_loader, val_loader, num_epochs=15, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # (Опционально) early stopping
    best_val_loss = float('inf')
    patience = 3
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        # --- TRAIN ---
        model.train()
        train_loss_sum = 0.0
        train_bar = tqdm(train_loader, desc="Train", leave=False)
        for audio_batch, labels_concat, input_lengths, target_lengths in train_bar:
            audio_batch = audio_batch.to(device)
            labels_concat = labels_concat.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            logits = model(audio_batch)  # [T, B, C]
            log_probs = F.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss_sum / len(train_loader)

        # --- VAL ---
        model.eval()
        val_loss_sum = 0.0
        val_bar = tqdm(val_loader, desc="Val", leave=False)
        with torch.no_grad():
            for audio_batch, labels_concat, input_lengths, target_lengths in val_bar:
                audio_batch = audio_batch.to(device)
                labels_concat = labels_concat.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(audio_batch)
                log_probs = F.log_softmax(logits, dim=2)
                loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
                val_loss_sum += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss_sum / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # optional early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_deep_ctc.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping!")
                break

############################################
# 8) Main
############################################
if __name__ == "__main__":
    df = pd.read_csv("train.csv")  # твой датасет
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    # Создаём датасеты
    # Обрати внимание, что мы не делаем аугментации,
    # только увеличиваем спектр (n_mels=128, hop_length=256)
    def transform_fn(y, sr=16000):
        return audio_to_melspectrogram(
            y, sr=sr, n_mels=128, n_fft=1024, hop_length=256
        )

    train_dataset = MorseAudioCTCDataset(train_df, sr=16000, transform=transform_fn)
    val_dataset   = MorseAudioCTCDataset(val_df, sr=16000, transform=transform_fn)

    # Даталоадеры
    # pool_time_factor=8, т.к. 3 пула => /2 * /2 * /2 = /8
    train_loader = DataLoader(
        train_dataset,
        batch_size=4,  # увеличить, если позволяет память
        shuffle=True,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=8)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=8)
    )

    # Модель
    model = DeepCTCModel(
        num_classes=45,
        in_channels=1,
        n_mels=128,
        hidden_size=256,
        lstm_layers=2,   # можно 3, но будет медленнее
        dropout=0.2      # можно 0.3, если есть риск переобучения
    )

    # Быстрый тест одного батча
    audio_batch, labels_concat, input_lengths, target_lengths = next(iter(train_loader))
    print("audio_batch shape:", audio_batch.shape)
    print("labels_concat shape:", labels_concat.shape)
    print("input_lengths:", input_lengths)
    print("target_lengths:", target_lengths)

    # Обучение
    train_ctc_loop(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=21,   # или больше, если нужно
        lr=1e-3
    )


audio_batch shape: torch.Size([4, 1, 128, 501])
labels_concat shape: torch.Size([37])
input_lengths: tensor([62, 62, 62, 62])
target_lengths: tensor([11, 10,  9,  7])

Epoch 1/21


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "Balanced.pth")